KHÔNG THÀNH CÔNG DO TenSEAL KHÔNG HỖ TRỢ PHÉP XOAY VÒNG ROTATE

In [2]:
pip install tenseal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.5 MB/s eta 0:00:00


In [116]:
import tenseal as ts
print(ts.__version__)

0.3.16


### test thu vien

In [51]:
import tenseal as ts

# Setup TenSEAL context
context = ts.context(
            ts.SCHEME_TYPE.CKKS,
            poly_modulus_degree=8192,
            coeff_mod_bit_sizes=[60, 40, 40, 60]
          )
context.generate_galois_keys()
context.global_scale = 2**40
number_of_slots = 8192 / 2

In [12]:
v1 = [0, 1, 2, 3, 4]
v2 = [4, 3, 2, 1, 0]

# encrypted vectors
enc_v1 = ts.ckks_vector(context, v1)
enc_v2 = ts.ckks_vector(context, v2)

result = enc_v1 + enc_v2
result.decrypt() # ~ [4, 4, 4, 4, 4]

result = enc_v1.dot(enc_v2)
result.decrypt() # ~ [10]

# matrix = [
#   [73, 0.5, 8],
#   [81, -5, 66],
#   [-100, -78, -2],
#   [0, 9, 17],
#   [69, 11 , 10],
# ]
# result = enc_v1.matmul(matrix)
# result.decrypt() # ~ [157, -90, 153]

[10.000001815515304]

###

In [65]:
path="/content/drive/MyDrive/Colab Notebooks/decision-tree-tenseal"

number_threshold = 31  # trong thực tế N x k = 70 x 4 = 280
max_depth = 2
ALL_SAMPLES = 10
NUM_SAMPLES_TRAIN = int(ALL_SAMPLES * 0.8)
NUM_SAMPLES_TEST  = ALL_SAMPLES - NUM_SAMPLES_TRAIN
num_label = 3
src_data = "/content/drive/MyDrive/Colab Notebooks/decision-tree-pyfhel/Release/iris_8_2.csv"
src_model_tree = "/content/drive/MyDrive/Colab Notebooks/decision-tree-pyfhel/model_tree/model_tree_ten.bin"

SOFT_STEP_COEFFICIENTS_16 = {
    5.00000000e-01,
    2.11445799e+00,
    1.15591931e-10,
    -6.38009501e+00,
    -4.91650318e-10,
    1.09534390e+01,
    8.95471329e-10,
    -1.01295272e+01,
    -8.38669100e-10,
    5.27558906e+00,
    4.36355800e-10,
    -1.54908047e+00,
    -1.27390646e-10,
    2.39094701e-01,
    1.95169389e-11,
    -1.50730122e-02,
    -1.22081599e-12
};
SOFT_STEP_COEFFICIENTS_8 = {
    0.5,
    1.23986659,
    -0.0,
    -1.05984904,
    -0.0,
    0.40068769,
    0.0,
    -0.05021892,
    0.0
};

In [66]:
import tenseal as ts

context = ts.context(
    scheme=ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=2**15,  # 32768
    coeff_mod_bit_sizes=[
        60,
        40, 40, 40, 40, 40, 40, 40, 40, 40,
        40, 40, 40, 40, 40, 40, 40, 40
    ]
)

context.generate_galois_keys()
context.global_scale = 2**40
number_of_slots = (2**15) / 2

### save & load keys thành công

In [20]:
with open(path+"/context_full.bin", "wb") as f:
    f.write(context.serialize())

In [21]:
with open(path+"/context_public.bin", "wb") as f:
    f.write(
        context.serialize(
            save_secret_key=False
        )
    )


In [22]:
with open(path+"/context_full.bin", "rb") as f:
    context = ts.context_from(f.read())

### Chuẩn bị và mã hóa dữ liệu

In [74]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [75]:
df = pd.read_csv(path+"/data/iris1.csv")
df.head()

,5.1,3.5,1.4,0.2,setosa
0,4.9,3.0,1.4,0.2,setosa
1,4.7,3.2,1.3,0.2,setosa
2,7.0,3.2,4.7,1.4,versicolor
3,6.4,3.2,4.5,1.5,versicolor
4,5.8,2.7,5.1,1.9,virginica


In [76]:
X = df.iloc[:, :-1].values     # tất cả cột trừ cột cuối
labels = df.iloc[:, -1].values
print(X[0])
print(labels[0])

[4.9 3.  1.4 0.2]
setosa


In [105]:
num_feature = X[0].size
print(num_feature)

4


In [77]:
scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)
print(X[0])

[-0.76        0.55555556 -0.95652174 -1.        ]


In [78]:
label_map = {
    "setosa": 0,
    "versicolor": 1,
    "virginica": 2
}

y_int = np.array([label_map[l] for l in labels])
print(y_int[:5])

[0 0 1 1 2]


In [79]:
num_classes = len(label_map)

Y = np.zeros((len(y_int), num_classes))
Y[np.arange(len(y_int)), y_int] = 1
print(Y[:5])

[[1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [80]:
X_train = X[:NUM_SAMPLES_TRAIN]
y_train = Y[:NUM_SAMPLES_TRAIN]
X_test = X[NUM_SAMPLES_TRAIN:]
y_test = Y[NUM_SAMPLES_TRAIN:]

In [97]:
C_X_cols = []

for i in range(X_train.shape[1]): # X_train.shape[1] là số lượng đặc trưng (features) của mỗi mẫu
    col = X_train[:, i] # lấy toàn bộ giá trị của cột thứ i trong ma trận X_train
    ctxt = ts.ckks_vector(context, col)
    C_X_cols.append(ctxt)

C_Y_cols = []

for i in range(y_train.shape[1]):
    col = y_train[:, i]
    ctxt = ts.ckks_vector(context, col)
    C_Y_cols.append(ctxt)

C_X_cells_test = []

for i in range(X_test.shape[0]):        # từng mẫu
    row_ctxts = []
    for j in range(X_test.shape[1]):    # từng feature
        value = X_test[i, j]
        ctxt = ts.ckks_vector(context, np.array([value]))
        row_ctxts.append(ctxt)
    C_X_cells_test.append(row_ctxts)

Mã hóa trọng số

In [98]:
vector_ones = np.ones(NUM_SAMPLES_TRAIN)  # kích thước 1D
C_W_col = ts.ckks_vector(context, vector_ones)

Mã hóa ngưỡng

In [99]:
min_T = -1.0
max_T = 1.0

step = (max_T - min_T - 0.4) / (number_threshold + 1)

all_thresholds = []

for i in range(1, number_threshold + 1):
    theta = min_T + i * step
    if -0.2 <= theta <= 0.2:
        continue
    if theta >= 1.0:
        break
    all_thresholds.append(theta)

number_threshold_real = len(all_thresholds)
print("Thresholds:", all_thresholds)
print("Number threshold real:", number_threshold_real)

C_thresholds = []

for t in all_thresholds:
    ctxt = ts.ckks_vector(context, np.array([t]))
    C_thresholds.append(ctxt)

Thresholds: [-0.95, -0.9, -0.85, -0.8, -0.75, -0.7, -0.6499999999999999, -0.6, -0.55, -0.5, -0.44999999999999996, -0.3999999999999999, -0.35, -0.29999999999999993, -0.25, 0.20000000000000018, 0.25, 0.30000000000000004, 0.3500000000000001, 0.40000000000000013, 0.4500000000000002, 0.5, 0.55]
Number threshold real: 23


Mã hóa feature

In [100]:
num_feature = X_train.shape[1]
C_I_one_hot = []
for i in range(num_feature):
    vec = np.zeros(num_feature)
    vec[i] = 1.0
    x = ts.ckks_vector(context, vec)
    C_I_one_hot.append(x)

### Hàm train & predict

In [54]:
class NodeC:
    def __init__(self):
        self.is_leaf = False
        # Nếu là lá: list PyCtxt, mỗi ciphertext chứa trọng số cho nhãn l
        self.leaf_value_vector = []

        # Nếu không là lá: phân chia
        self.feature_index = None
        self.threshold = None    # PyCtxt

        # Node con
        self.left_child = None
        self.right_child = None

In [84]:
def sum_slots(ct: ts.CKKSVector, n_slots: int, ts):
    result = ct
    step = 1
    while step < n_slots:
        result = result + result.rotate(step)
        step <<= 1
    return result

def mul_cipher_with_cipher_slot0(ct: ts.CKKSVector,
                                 ct_ref: ts.CKKSVector,
                                 n_slots: int,
                                 ts):
    """
    Multiply ct with slot-0 value of ct_ref
    """

    # Mask giữ slot 0
    mask = [1.0] + [0.0] * (n_slots - 1)
    ct_slot0 = ct_ref * mask

    # Broadcast slot 0
    step = 1
    while step < n_slots:
        ct_slot0 = ct_slot0 + ct_slot0.rotate(step)
        step <<= 1

    # Multiply
    return ct * ct_slot0


In [86]:
def soft_step_evaluation(encrypted_z, SOFT_STEP_COEFFICIENTS, number_of_slots, ts):
    """
    encrypted_z: PyCtxt, ciphertext của z = cx - theta
    HE: Pyfhel object
    SOFT_STEP_COEFFICIENTS: list hệ số đa thức c0, c1, ...
    """
    # 1. Tạo vector lưu các lũy thừa của z: z^1, z^2, z^4, z^8, ... (exponentiation by squaring)
    powers = [encrypted_z]  # z^1
    i = 1
    while i < len(SOFT_STEP_COEFFICIENTS):
        z_for_mul = powers[-1]
        tmp = powers[-1] + z_for_mul
        powers.append(tmp)
        i *= 2

    # 2. Encode c0 và khởi tạo result
    c0 = SOFT_STEP_COEFFICIENTS[0]
    result = ts.ckks_vector(
        context,
        [c0] * number_of_slots
    )

    # 3. Tính đa thức: result = c0 + c1*z + c2*z^2 + ...
    for i in range(1, len(SOFT_STEP_COEFFICIENTS)):
        coeff = SOFT_STEP_COEFFICIENTS[i]
        if abs(coeff) < 1e-12:
            continue

        # 3.1 Tính z^i bằng cách kết hợp các lũy thừa 2^j
        power = None
        first = True
        j = 0
        while (1 << j) <= i:
            if i & (1 << j):
                if first:
                    power = powers[j]
                    first = False
                else:
                    power = power*powers[j]
            j += 1

        term = power * coeff
        result = result + term

    return result


In [55]:
def leaf_value(C_W_col, C_Y_cols, num_label, number_of_slots, ts):
    """
    C_W_col: PyCtxt của trọng số cột W
    C_Y_cols: list PyCtxt của nhãn one-hot (column-wise)
    HE: Pyfhel object
    num_label: số nhãn
    return: list PyCtxt, mỗi element = tổng trọng số cho nhãn l
    """
    print("leaf_value()")
    C_leaf_values = []

    for l in range(num_label):
        print(f"\tProcessing label {l}")
        P = C_W_col*C_Y_cols[l]

        # Nếu muốn tính tổng slots, dùng sum_slots(P, HE, n_slots)
        P = sum_slots(P, number_of_slots, ts)

        C_leaf_values.append(P)

    print("Completed leaf_value()")
    return C_leaf_values

In [113]:
def compute_weighted_counts_homo(best_feature, C_T_col, C_X_cols, C_W_col, C_Y_cols,
                                 num_feature, num_label, SOFT_STEP_COEFFICIENTS, ts):
    """
    best_feature: list PyCtxt, 1 value per feature (plaintext broadcasted to ciphertext)
    C_T_col: PyCtxt (threshold)
    C_X_cols: list PyCtxt, K features
    C_W_col: PyCtxt, trọng số
    C_Y_cols: list PyCtxt, L nhãn
    HE: Pyfhel object
    num_feature: số feature
    num_label: số nhãn
    SOFT_STEP_COEFFICIENTS: list hệ số soft-step polynomial
    """
    print("compute_weighted_counts_homo()")
    C_right_counts = [None] * num_label
    C_left_counts = [None] * num_label

    # Tính C_X[i] = sum_k(best_feature[k] * C_X_cols[k])
    print("\tTính C_X[i]")
    C_X_i = None
    first = True
    for k in range(num_feature):
        x = best_feature.rotate(k)
        term = mul_cipher_with_cipher_slot0(C_X_cols[k], x, number_of_slots, ts)
        if C_X_i is None:
            C_X_i = term
            first = False
        else:
            C_X_i = C_X_i+term

    # Tính Z_right = C_X[i] - C_T_col, Z_left = C_T_col - C_X[i]
    print("\tTính Z_right và Z_left")
    C_Z_right = C_X_i-C_T_col
    C_Z_left = C_T_col-C_X_i

    # Soft-step evaluation
    print("\tTính soft-step()")
    C_Phi_Right = soft_step_evaluation(C_Z_right, SOFT_STEP_COEFFICIENTS, ts)
    C_Phi_Left  = soft_step_evaluation(C_Z_left,  SOFT_STEP_COEFFICIENTS, ts)

    # Nhân W * Phi
    print("\tNhân C_W_col * Phi")
    C_W_Phi_Right = C_W_col*C_Phi_Right
    C_W_Phi_Left  = C_W_col*C_Phi_Left

    # Tính weighted counts per label
    print("\tCompute weighted counts per label")
    for l in range(num_label):
        # RIGHT
        C_term_right = C_W_Phi_Right*C_Y_cols[l]
        C_right_counts[l] = sum_slots(C_term_right, number_of_slots)
        # LEFT
        C_term_left = C_W_Phi_Left*C_Y_cols[l]
        C_left_counts[l] = sum_slots(C_term_left, number_of_slots)

    print("Completed compute_weighted_counts_homo()")

    return C_right_counts, C_left_counts

In [88]:
def compute_gini_impurity(right_counts_, left_counts_, NUM_SAMPLES_TRAIN):
    """
    right_counts_: list of list of float, mỗi list tương ứng 1 nhãn (Lx1)
    left_counts_: list of list of float
    NUM_SAMPLES_TRAIN: tổng số mẫu train, dùng để sum slots nếu cần
    return: float, gini impurity
    """
    print("compute_gini_impurity()")
    # Sum slots nếu input là list per sample
    right_counts = [counts[0] for counts in right_counts_]
    left_counts  = [counts[0] for counts in left_counts_]

    # Tổng trọng số
    total_right = sum(right_counts)
    total_left  = sum(left_counts)
    total_all   = total_right + total_left
    if total_all < 1e-9:
        return 0.0

    # Gini bên phải
    gini_right = 0.0
    if total_right > 1e-9:
        sum_sq = sum((count / total_right)**2 for count in right_counts)
        gini_right = (1.0 - sum_sq) * (total_right / total_all)

    # Gini bên trái
    gini_left = 0.0
    if total_left > 1e-9:
        sum_sq = sum((count / total_left)**2 for count in left_counts)
        gini_left = (1.0 - sum_sq) * (total_left / total_all)

    print("Completed compute_gini_impurity()")
    return gini_right + gini_left

In [58]:
def compute_W_phi_best(best_feature, best_threshold, C_X_cols, C_W_col,
                       num_feature, SOFT_STEP_COEFFICIENTS, number_of_slots, ts):
    """
    best_feature: list PyCtxt one-hot
    best_threshold: PyCtxt (threshold)
    C_X_cols: list PyCtxt các feature
    C_W_col: PyCtxt trọng số
    HE: Pyfhel object
    num_feature: số feature
    SOFT_STEP_COEFFICIENTS: list hệ số soft-step polynomial
    return: tuple (C_W_new_right, C_W_new_left)
    """
    print("compute_W_phi_best()")

    # Tính X[best_feature] = sum_k(best_feature[k] * C_X_cols[k])
    print("\tTính X[best_feature]")
    C_X_i = None
    first = True
    for k in range(num_feature):
        x = best_feature[k].rotate(k)
        term = mul_cipher_with_cipher_slot0(C_X_cols[k], x, number_of_slots)
        if C_X_i is None:
            C_X_i = term
            first = False
        else:
            C_X_i = C_X_i+term

    # Z_right = X - theta, Z_left = theta - X
    print("\tTính Z = X[best_feature] - theta")
    C_Z_right = C_X_i-best_threshold
    C_Z_left  = best_threshold-C_X_i

    # Soft-step evaluation
    print("\tTính soft-step(Z)")
    C_Phi_Right = soft_step_evaluation(C_Z_right, SOFT_STEP_COEFFICIENTS, number_of_slots, ts)
    C_Phi_Left  = soft_step_evaluation(C_Z_left,  SOFT_STEP_COEFFICIENTS, number_of_slots, ts)

    # W_new = W * Phi
    print("\tTính W_new = W * Phi")
    C_W_new_right = C_W_col*C_Phi_Right
    C_W_new_left  = C_W_col*C_Phi_Left

    print("Completed compute_W_phi_best()")
    return C_W_new_right, C_W_new_left

In [89]:
def train_decision_tree(C_X_cols, C_W_col, C_Y_cols, C_T_cols, depth, max_depth,
                        num_feature, C_I_one_hot, NUM_SAMPLES_TRAIN, num_label,
                        SOFT_STEP_COEFFICIENTS, number_of_slots, ts):
    print(f"train_decision_tree() depth={depth}")
    node_c = NodeC()

    # Điều kiện dừng
    if depth >= max_depth:
        print(f"Node leaf tại depth={depth}")
        node_c.is_leaf = True
        C_leaf_values = leaf_value(C_W_col, C_Y_cols, num_label, number_of_slots, ts)
        node_c.leaf_value_vector = C_leaf_values
        return node_c

    # Tính weighted counts cho từng feature và threshold
    print("Tính weighted counts cho từng feature và threshold")
    C_right_counts_C_T_cols_I = []
    C_left_counts_C_T_cols_I  = []

    for i in range(num_feature):
        print(f"\tFeature {i}")
        C_one_hot_feature = C_I_one_hot[i]
        C_right_counts_C_T_cols = []
        C_left_counts_C_T_cols  = []

        for j, C_T_col in enumerate(C_T_cols):
            print(f"\t\tThreshold {j}")
            C_right_counts, C_left_counts = compute_weighted_counts_homo(
                C_one_hot_feature, C_T_col, C_X_cols, C_W_col, C_Y_cols,
                num_feature, num_label, SOFT_STEP_COEFFICIENTS, ts
            )
            C_right_counts_C_T_cols.append(C_right_counts)
            C_left_counts_C_T_cols.append(C_left_counts)

        C_right_counts_C_T_cols_I.append(C_right_counts_C_T_cols)
        C_left_counts_C_T_cols_I.append(C_left_counts_C_T_cols)

    # Client decrypt và tính Gini, tìm best feature & threshold
    print("Decrypt và tính Gini, chọn best_feature & best_threshold")
    min_gini = 1e9
    best_feature_idx = -1
    best_threshold_idx = -1
    num_theta = len(C_T_cols)

    for i in range(num_feature):
        for j in range(num_theta):
            # Decrypt counts cho nhãn
            right_counts_clear = []
            left_counts_clear  = []

            for l in range(num_label):
                decoded_r = C_right_counts_C_T_cols_I[i][j][l].decrypt()
                decoded_l = C_left_counts_C_T_cols_I[i][j][l].decrypt()
                right_counts_clear.append(decoded_r)
                left_counts_clear.append(decoded_l)

            # Tính Gini
            current_gini = compute_gini_impurity(right_counts_clear, left_counts_clear, NUM_SAMPLES_TRAIN, ts)
            if current_gini < min_gini:
                min_gini = current_gini
                best_feature_idx = i
                best_threshold_idx = j

    print(f"best_feature={best_feature_idx}, best_threshold={best_threshold_idx}")

    # Lấy ciphertext best_feature one-hot & best_threshold
    C_best_feature = C_I_one_hot[best_feature_idx]
    C_best_threshold = C_T_cols[best_threshold_idx]

    node_c.feature_index = C_best_feature
    node_c.threshold = C_best_threshold

    # Tính W_new cho nhánh trái & phải
    print("Tính W_new cho de quy")
    C_W_new_right, C_W_new_left = compute_W_phi_best(
        C_best_feature, C_best_threshold, C_X_cols, C_W_col,
        num_feature, SOFT_STEP_COEFFICIENTS, number_of_slots, ts
    )

    # Recursively train children
    node_c.right_child = train_decision_tree(
        C_X_cols, C_W_new_right, C_Y_cols, C_T_cols, depth+1, max_depth
        , num_feature, C_I_one_hot, NUM_SAMPLES_TRAIN, num_label, SOFT_STEP_COEFFICIENTS, number_of_slots, ts
    )
    node_c.left_child = train_decision_tree(
        C_X_cols, C_W_new_left, C_Y_cols, C_T_cols, depth+1, max_depth
        , num_feature, C_I_one_hot, NUM_SAMPLES_TRAIN, num_label, SOFT_STEP_COEFFICIENTS, number_of_slots, ts
    )

    print(f"Completed train_decision_tree() at depth={depth}")
    return node_c

In [90]:
def predict_decision_tree(node_c, C_X_cols, num_feature, num_label, SOFT_STEP_COEFFICIENTS, number_of_slots, ts):
    print("predict_decision_tree()")

    # Nếu là node leaf
    if node_c.is_leaf:
        print("Reached leaf node. Returning leaf values.")
        return node_c.leaf_value_vector

    # Không phải leaf — tính soft-step
    i_best = node_c.feature_index      # list PyCtxt one-hot
    C_Theta = node_c.threshold         # PyCtxt

    # Tính X_i = sum_k i_best[k] * C_X_cols[k]
    C_X_i = None
    first = True
    for k in range(num_feature):
        x = i_best[k].rotate(k)
        term = mul_cipher_with_cipher_slot0(C_X_cols[k], x, number_of_slots)
        if C_X_i is None:
            C_X_i = term
            first = False
        else:
            C_X_i = C_X_i+term

    # Tính Z_right = X_i - Theta, Z_left = Theta - X_i
    C_Z_right = C_X_i-C_Theta
    C_Z_left  = C_Theta-C_X_i

    # Tính soft-step
    C_Phi_Right = soft_step_evaluation(C_Z_right, C_Phi_Right, number_of_slots, ts)
    C_Phi_Left  = soft_step_evaluation(C_Z_left,  C_Phi_Left,  number_of_slots, ts)

    # Đệ quy
    C_Output_Right = predict_decision_tree(node_c.right_child, C_X_cols, num_feature, num_label, SOFT_STEP_COEFFICIENTS, number_of_slots, ts)
    C_Output_Left  = predict_decision_tree(node_c.left_child,  C_X_cols, num_feature, num_label, SOFT_STEP_COEFFICIENTS, number_of_slots, ts)

    # Nhân soft-step với output từng nhánh
    C_Out_Phi_Right = []
    C_Out_Phi_Left  = []
    for l in range(num_label):
        C_Out_Phi_Right.append(C_Output_Right[l]*C_Phi_Right)
        C_Out_Phi_Left.append(C_Output_Left[l]*C_Phi_Left)

    # Tổng hợp kết quả
    C_Final_Output = []
    for l in range(num_label):
        C_Final_Output.append(C_Out_Phi_Right[l]+C_Out_Phi_Left[l])

    print("Completed predict_decision_tree()")
    return C_Final_Output


In [91]:
import json
from typing import List
import numpy as np

# Hàm tính accuracy
def calculate_accuracy(decoded_predictions: List[List[float]], Y_test_onehot: List[List[float]]) -> float:
    print("calculate_accuracy()")
    NUM_LABELS = len(decoded_predictions)
    NUM_SAMPLES_TEST = len(Y_test_onehot)
    correct = 0

    for i in range(NUM_SAMPLES_TEST):
        pred_scores = [decoded_predictions[l][i] for l in range(NUM_LABELS)]
        pred_label = int(np.argmax(pred_scores))

        true_label = int(np.argmax(Y_test_onehot[i]))

        if pred_label == true_label:
            correct += 1

    acc = correct / NUM_SAMPLES_TEST
    print(f"Accuracy: {acc:.6f}")
    return acc

# Hàm tính Macro F1-Score
def calculate_f1_score(decoded_predictions: List[List[float]], Y_test_onehot: List[List[float]]) -> float:
    print("calculate_f1_score() - Macro Average")
    L = len(decoded_predictions)
    N = len(Y_test_onehot)

    true_labels = [int(np.argmax(Y_test_onehot[i])) for i in range(N)]
    predicted_labels = [int(np.argmax([decoded_predictions[l][i] for l in range(L)])) for i in range(N)]

    # Confusion Matrix
    CM = np.zeros((L, L), dtype=int)
    for t, p in zip(true_labels, predicted_labels):
        CM[t][p] += 1

    f1_scores = []
    print("\n--- F1 Score Per Class ---")
    print("Class | Precision | Recall | F1-Score")
    for l in range(L):
        TP = CM[l][l]
        FP = sum(CM[:, l]) - TP
        FN = sum(CM[l, :]) - TP
        precision = 0.0 if TP + FP == 0 else TP / (TP + FP)
        recall    = 0.0 if TP + FN == 0 else TP / (TP + FN)
        f1 = 0.0 if precision + recall < 1e-9 else 2 * (precision * recall) / (precision + recall)
        f1_scores.append(f1)
        print(f"{l:5} | {precision:9.4f} | {recall:6.4f} | {f1:8.4f}")

    macro_f1 = sum(f1_scores) / L if L > 0 else 0.0
    print(f"\nMacro F1-Score (Average): {macro_f1:.6f}")
    return macro_f1

# Hàm in cây đã giải mã
def print_tree_decrypted(node_c, ts, depth=0):
    if node_c is None:
        return

    indent = "  " * depth

    if node_c.is_leaf:
        print(indent + "Leaf Node: [", end="")
        vals = []
        for ctxt in node_c.leaf_value_vector:
            pt = ctxt.decrypt()
            vals.append(sum(pt))
        print(", ".join(f"{v:.4f}" for v in vals) + "]")
        return

    # Internal node: decode feature_index one-hot -> index
    feature_index = -1
    for i, ctxt in enumerate(node_c.feature_index):
        val = ctxt.decrypt()[0]
        if val > 0.5:
            feature_index = i
            break

    threshold = node_c.threshold.decrypt()[0]
    print(indent + f"Internal Node: Feature Index = {feature_index}, Threshold = {threshold:.4f}")

    print_tree_decrypted(node_c.left_child, ts, depth+1)
    print_tree_decrypted(node_c.right_child, ts, depth+1)

### Thuc nghiem

In [92]:
import time

In [114]:
root = train_decision_tree(
    C_X_cols, C_W_col, C_Y_cols, C_thresholds,
    depth=0,
    max_depth=max_depth,
    num_feature=num_feature,
    C_I_one_hot=C_I_one_hot,
    NUM_SAMPLES_TRAIN=NUM_SAMPLES_TRAIN,
    num_label=num_label,
    SOFT_STEP_COEFFICIENTS=SOFT_STEP_COEFFICIENTS_16,
    number_of_slots=number_of_slots,
    ts=ts
)

train_decision_tree() depth=0
Tính weighted counts cho từng feature và threshold
	Feature 0
		Threshold 0
compute_weighted_counts_homo()
	Tính C_X[i]


AttributeError: 'CKKSVector' object has no attribute 'rotate'